# HIT140 Foundations of Data Science — Assignment 2
## Positional Shooting Precision
**Analytic question:** Do Forwards record a significantly higher average of Shots on Target per 90 minutes than Midfielders?

Run the notebooks in numerical order. Each notebook reads the CSV exported by the preceding stage, so the workflow remains reproducible.

# Notebook 4 — Inferential Statistics: 95% Confidence Intervals
This notebook estimates a 95% confidence interval for the mean SoT/90 of each position group. Because the population standard deviations are unknown, the interval uses the **t distribution** with `n − 1` degrees of freedom. The cleaned population CSV is also loaded only to check whether the known dataset population mean falls inside each sample-based interval.


In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as st
import os

sample_file = "02_sampled_player_data.csv"
population_file = "01_wrangled_player_data.csv"

for file in [sample_file, population_file]:
    if not os.path.exists(file):
        raise FileNotFoundError(f"Required file not found: {file}")

sample_df = pd.read_csv(sample_file)
population_df = pd.read_csv(population_file)


In [ ]:
def mean_ci_t(values, confidence=0.95):
    values = values.dropna()
    n = len(values)
    mean = values.mean()
    std = values.std(ddof=1)
    se = std / np.sqrt(n)
    t_critical = st.t.ppf((1 + confidence) / 2, df=n - 1)
    margin = t_critical * se
    return {
        "n": n,
        "mean": mean,
        "std": std,
        "standard_error": se,
        "t_critical": t_critical,
        "ci_lower": mean - margin,
        "ci_upper": mean + margin
    }

rows = []
for position in ["Forward", "Midfielder"]:
    sample_values = sample_df.loc[
        sample_df["Position_Group"] == position, "SoT/90"
    ]
    result = mean_ci_t(sample_values)

    population_mean = population_df.loc[
        population_df["Position_Group"] == position, "SoT/90"
    ].mean()

    result["Position_Group"] = position
    result["population_mean"] = population_mean
    result["population_mean_enclosed"] = (
        result["ci_lower"] <= population_mean <= result["ci_upper"]
    )
    rows.append(result)

ci_results = pd.DataFrame(rows).set_index("Position_Group")

print("95% Confidence Intervals for Mean SoT/90")
display(ci_results.round(4))

ci_results.to_csv("04_confidence_interval_results.csv", encoding="utf-8-sig")


### Interpretation reminder
A 95% confidence interval gives a plausible range for the population mean under the sampling procedure. It should not be described as a 95% probability that the already-computed fixed interval contains the true mean. Across repeated samples, approximately 95% of intervals constructed this way would contain the true population mean.
